In [ ]:
#Chamando pacotes

import numpy as np
import pandas as pd
import scipy.stats as stats
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    adjusted_rand_score, silhouette_score, confusion_matrix
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
# Estilo visual para os gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 11})

# ==============================================================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO DOS DADOS (SPAMBASE)
# ==============================================================================
data_path = "spambase.data"

feature_names = [f"word_freq_{i}" for i in range(48)] + \
                [f"char_freq_{i}" for i in range(6)] + \
                ["capital_run_length_average", "capital_run_length_longest", "capital_run_length_total"]

col_names = feature_names + ["is_spam"]

# Leitura do dataset
df = pd.read_csv(data_path, header=None, names=col_names)

# Separação em matriz de características X e rótulos y_true
X_raw = df.drop(columns=["is_spam"]).values
y_true = df["is_spam"].values

# Padronização global dos dados
scaler_global = StandardScaler()
X = scaler_global.fit_transform(X_raw)


# ==============================================================================
# 2. IMPLEMENTAÇÃO DO DOUBLE K-MEANS (QUESTÃO 1)
# ==============================================================================
def double_kmeans(X, K, H, max_iter=100, tol=1e-6, random_state=None):
    rng = np.random.RandomState(random_state)
    N, P = X.shape

    u_labels = rng.randint(0, K, size=N)
    v_labels = rng.randint(0, H, size=P)

    for k in range(K):
        if not np.any(u_labels == k):
            u_labels[rng.choice(N)] = k
    for h in range(H):
        if not np.any(v_labels == h):
            v_labels[rng.choice(P)] = h

    G = np.zeros((K, H))
    W_history = []

    for iteration in range(max_iter):
        for k in range(K):
            idx_obj = np.where(u_labels == k)[0]
            for h in range(H):
                idx_var = np.where(v_labels == h)[0]
                if len(idx_obj) > 0 and len(idx_var) > 0:
                    G[k, h] = np.mean(X[np.ix_(idx_obj, idx_var)])
                else:
                    G[k, h] = 0.0

        dist_u = np.zeros((N, K))
        for h in range(H):
            idx_var = np.where(v_labels == h)[0]
            if len(idx_var) == 0:
                continue
            X_sub = X[:, idx_var]
            for k in range(K):
                dist_u[:, k] += np.sum((X_sub - G[k, h]) ** 2, axis=1)

        new_u_labels = np.argmin(dist_u, axis=1)

        for k in range(K):
            if not np.any(new_u_labels == k):
                worst_obj = np.argmax(np.min(dist_u, axis=1))
                new_u_labels[worst_obj] = k

        dist_v = np.zeros((P, H))
        for k in range(K):
            idx_obj = np.where(new_u_labels == k)[0]
            if len(idx_obj) == 0:
                continue
            X_sub = X[idx_obj, :]
            for h in range(H):
                dist_v[:, h] += np.sum((X_sub - G[k, h]) ** 2, axis=0)

        new_v_labels = np.argmin(dist_v, axis=1)

        for h in range(H):
            if not np.any(new_v_labels == h):
                worst_var = np.argmax(np.min(dist_v, axis=1))
                new_v_labels[worst_var] = h

        W = 0.0
        for k in range(K):
            idx_obj = np.where(new_u_labels == k)[0]
            for h in range(H):
                idx_var = np.where(new_v_labels == h)[0]
                if len(idx_obj) > 0 and len(idx_var) > 0:
                    W += np.sum((X[np.ix_(idx_obj, idx_var)] - G[k, h]) ** 2)

        W_history.append(W)

        if np.array_equal(new_u_labels, u_labels) and np.array_equal(new_v_labels, v_labels):
            u_labels, v_labels = new_u_labels, new_v_labels
            break
        if iteration > 0 and abs(W_history[-2] - W_history[-1]) < tol:
            u_labels, v_labels = new_u_labels, new_v_labels
            break

        u_labels, v_labels = new_u_labels, new_v_labels

    return G, u_labels, v_labels, W, W_history


# Grid de execução do Double k-means
K_values = [2, 3, 4]
results = {}
grid_pairs = []

print("Iniciando varredura dos pares (K, H) com 100 repetições cada...")
for K in K_values:
    for H in range(1, K + 1):
        pair = (K, H)
        grid_pairs.append(pair)
        best_W = np.inf
        best_run = None

        for run in range(100):
            G, u_labels, v_labels, W, W_hist = double_kmeans(
                X, K=K, H=H, max_iter=60, random_state=run * 100 + K * 10 + H
            )
            if W < best_W:
                best_W = W
                best_run = {
                    "G": G, "u_labels": u_labels, "v_labels": v_labels,
                    "W": W, "W_history": W_hist
                }

        sil = silhouette_score(X, best_run["u_labels"])
        best_run["silhouette"] = sil
        results[pair] = best_run
        print(f"Par (K={K}, H={H}) -> Menor W = {best_W:.2f} | Silhueta = {sil:.4f}")

sil_values = [results[pair]["silhouette"] for pair in grid_pairs]
best_pair = grid_pairs[np.argmax(sil_values)]
K_star = best_pair[0]

print("\n" + "="*50)
print(f"Melhor combinação (K, H): {best_pair}")
print(f"Número ótimo de clusters de objetos K* = {K_star} (Silhueta máxima = {max(sil_values):.4f})")
print("="*50 + "\n")

optimal_run = results[best_pair]
u_star = optimal_run["u_labels"]
G_star = optimal_run["G"]
W_history_star = optimal_run["W_history"]

# Índice de Rand Corrigido (ARI)
ari = adjusted_rand_score(y_true, u_star)
print(f"Índice de Rand Corrigido (ARI) para K* = {K_star}: {ari:.4f}")


# ==============================================================================
# 3. DEFINIÇÃO DOS CLASSIFICADORES CUSTOMIZADOS (QUESTÃO 2)
# ==============================================================================

class GaussianBayesClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_ = {}
        self.means_ = {}
        self.covs_ = {}
        n_samples, self.d_ = X.shape

        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / n_samples
            self.means_[c] = np.mean(X_c, axis=0)
            cov = np.cov(X_c, rowvar=False) + 1e-4 * np.eye(self.d_)
            self.covs_[c] = cov
        return self

    def predict_proba(self, X):
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        log_posteriors = np.zeros((n_samples, n_classes))

        for idx_c, c in enumerate(self.classes_):
            prior = self.priors_[c]
            mean = self.means_[c]
            cov = self.covs_[c]
            
            try:
                log_pdf = stats.multivariate_normal.logpdf(X, mean=mean, cov=cov, allow_singular=True)
            except Exception:
                log_pdf = np.full(n_samples, -1e10)
                
            log_posteriors[:, idx_c] = log_pdf + np.log(prior + 1e-15)

        max_log = np.max(log_posteriors, axis=1, keepdims=True)
        exp_post = np.exp(log_posteriors - max_log)
        return exp_post / np.sum(exp_post, axis=1, keepdims=True)

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


class KNNBayesClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5, metric='euclidean'):
        self.n_neighbors = n_neighbors
        self.metric = metric

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.knn_ = KNeighborsClassifier(n_neighbors=self.n_neighbors, metric=self.metric)
        self.knn_.fit(X, y)
        return self

    def predict(self, X):
        return self.knn_.predict(X)

    def predict_proba(self, X):
        return self.knn_.predict_proba(X)


class ParzenBayesClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, h=1.0):
        self.h = h

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_ = {}
        self.X_by_class_ = {}
        n_samples = len(y)
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / n_samples
            self.X_by_class_[c] = X_c
        self.d_ = X.shape[1]
        return self

    def _parzen_pdf_class(self, X_eval, X_c):
        n_eval = X_eval.shape[0]
        pdf_vals = np.zeros(n_eval)
        coeff = 1.0 / (np.sqrt(2 * np.pi) * self.h)
        
        for i in range(n_eval):
            diff = (X_eval[i] - X_c) / self.h
            k_univ = coeff * np.exp(-0.5 * (diff ** 2))
            prod_kernel = np.prod(k_univ, axis=1)
            pdf_vals[i] = np.mean(prod_kernel)
            
        return pdf_vals

    def predict_proba(self, X):
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        posteriors = np.zeros((n_samples, n_classes))

        for idx_c, c in enumerate(self.classes_):
            X_c = self.X_by_class_[c]
            prior = self.priors_[c]
            pdf_vals = self._parzen_pdf_class(X, X_c)
            posteriors[:, idx_c] = pdf_vals * prior

        sum_post = np.sum(posteriors, axis=1, keepdims=True)
        sum_post[sum_post == 0] = 1e-15
        return posteriors / sum_post

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


# ==============================================================================
# 4. PIPELINE COMPLETO DE VALIDAÇÃO CRUZADA 30x10-FOLDS (SEM LEAKAGE)
# ==============================================================================
def run_full_q2_evaluation(X_data, y_labels, dataset_name="Original"):
    print(f"\n==================================================")
    print(f"INICIANDO EXPERIMENTO 30x10-FOLDS PARA: {dataset_name}")
    print(f"==================================================")
    
    n_reps = 30
    n_splits = 10
    clf_names = ['Bayes_Gauss', 'Bayes_kNN', 'Bayes_Parzen', 'Reg_Logistica', 'Voto_Majoritario']
    metrics_list = ['Taxa_Erro', 'Precision', 'Recall', 'F1_Measure']
    
    results_matrix = {clf: {m: np.zeros(n_reps) for m in metrics_list} for clf in clf_names}
    
    train_fractions = np.linspace(0.05, 0.95, 19)
    learning_curves = {
        clf: {
            'train': np.zeros((n_reps, len(train_fractions))),
            'test': np.zeros((n_reps, len(train_fractions)))
        } for clf in clf_names
    }

    param_grids = {
        'Bayes_kNN': {
            'clf__n_neighbors': [1, 3, 5, 7, 9],
            'clf__metric': ['euclidean', 'manhattan', 'chebyshev']
        },
        'Bayes_Parzen': {
            'clf__h': [0.1, 0.5, 1.0, 2.0]
        },
        'Reg_Logistica': {
            'clf__C': [0.01, 0.1, 1.0, 10.0]
        }
    }

    for rep in range(n_reps):
        print(f"  -> Repetição {rep+1}/{n_reps}...")
        rkf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + rep)
        
        fold_metrics = {clf: {m: [] for m in metrics_list} for clf in clf_names}
        
        for train_idx, test_idx in rkf.split(X_data, y_labels):
            X_tr, X_te = X_data[train_idx], X_data[test_idx]
            y_tr, y_te = y_labels[train_idx], y_labels[test_idx]

            pipe_gauss = Pipeline([('scaler', StandardScaler()), ('clf', GaussianBayesClassifier())])
            pipe_gauss.fit(X_tr, y_tr)

            pipe_knn = Pipeline([('scaler', StandardScaler()), ('clf', KNNBayesClassifier())])
            grid_knn = GridSearchCV(pipe_knn, param_grids['Bayes_kNN'], cv=5, scoring='f1_macro', n_jobs=-1)
            grid_knn.fit(X_tr, y_tr)
            clf_knn = grid_knn.best_estimator_

            pipe_parzen = Pipeline([('scaler', StandardScaler()), ('clf', ParzenBayesClassifier())])
            grid_parzen = GridSearchCV(pipe_parzen, param_grids['Bayes_Parzen'], cv=5, scoring='f1_macro', n_jobs=-1)
            grid_parzen.fit(X_tr, y_tr)
            clf_parzen = grid_parzen.best_estimator_

            pipe_lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])
            grid_lr = GridSearchCV(pipe_lr, param_grids['Reg_Logistica'], cv=5, scoring='f1_macro', n_jobs=-1)
            grid_lr.fit(X_tr, y_tr)
            clf_lr = grid_lr.best_estimator_

            pred_gauss = pipe_gauss.predict(X_te)
            pred_knn = clf_knn.predict(X_te)
            pred_parzen = clf_parzen.predict(X_te)
            pred_lr = clf_lr.predict(X_te)

            all_preds = np.vstack([pred_gauss, pred_knn, pred_parzen, pred_lr])
            pred_voting, _ = stats.mode(all_preds, axis=0)
            pred_voting = pred_voting.ravel()

            predictions = {
                'Bayes_Gauss': pred_gauss,
                'Bayes_kNN': pred_knn,
                'Bayes_Parzen': pred_parzen,
                'Reg_Logistica': pred_lr,
                'Voto_Majoritario': pred_voting
            }

            for name, y_pred in predictions.items():
                err = 1.0 - accuracy_score(y_te, y_pred)
                prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
                rec = recall_score(y_te, y_pred, average='macro', zero_division=0)
                f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)

                fold_metrics[name]['Taxa_Erro'].append(err)
                fold_metrics[name]['Precision'].append(prec)
                fold_metrics[name]['Recall'].append(rec)
                fold_metrics[name]['F1_Measure'].append(f1)

        for name in clf_names:
            for m in metrics_list:
                results_matrix[name][m][rep] = np.mean(fold_metrics[name][m])

        X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(
            X_data, y_labels, test_size=0.3, stratify=y_labels, random_state=42 + rep
        )

        models_base = [
            Pipeline([('scaler', StandardScaler()), ('clf', GaussianBayesClassifier())]),
            Pipeline([('scaler', StandardScaler()), ('clf', KNNBayesClassifier(n_neighbors=5))]),
            Pipeline([('scaler', StandardScaler()), ('clf', ParzenBayesClassifier(h=1.0))]),
            Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])
        ]

        for frac_idx, frac in enumerate(train_fractions):
            if frac == 1.0:
                X_sub, y_sub = X_tr_full, y_tr_full
            else:
                X_sub, _, y_sub, _ = train_test_split(
                    X_tr_full, y_tr_full, train_size=frac, stratify=y_tr_full, random_state=42 + rep
                )
            
            sub_preds_tr = []
            sub_preds_te = []
            
            for m_idx, model in enumerate(models_base):
                model.fit(X_sub, y_sub)
                p_tr = model.predict(X_sub)
                p_te = model.predict(X_te_full)
                
                sub_preds_tr.append(p_tr)
                sub_preds_te.append(p_te)

                clf_name = clf_names[m_idx]
                learning_curves[clf_name]['train'][rep, frac_idx] = f1_score(y_sub, p_tr, average='macro', zero_division=0)
                learning_curves[clf_name]['test'][rep, frac_idx] = f1_score(y_te_full, p_te, average='macro', zero_division=0)

            vote_tr, _ = stats.mode(np.vstack(sub_preds_tr), axis=0)
            vote_te, _ = stats.mode(np.vstack(sub_preds_te), axis=0)
            
            learning_curves['Voto_Majoritario']['train'][rep, frac_idx] = f1_score(y_sub, vote_tr.ravel(), average='macro', zero_division=0)
            learning_curves['Voto_Majoritario']['test'][rep, frac_idx] = f1_score(y_te_full, vote_te.ravel(), average='macro', zero_division=0)

        gc.collect()

    return results_matrix, learning_curves, train_fractions


# ==============================================================================
# 5. EXECUÇÃO DA AVALIAÇÃO PARA AS DUAS VERSÕES DO DATASET
# ==============================================================================
results_orig, curves_orig, train_fractions = run_full_q2_evaluation(X_raw, y_true, "Spambase (Original - 2 Classes)")
results_clust, curves_clust, _ = run_full_q2_evaluation(X_raw, u_star, f"Spambase (Double k-means K*={K_star} Classes)")


# ==============================================================================
# 6. VISUALIZAÇÃO DOS DADOS E RESULTADOS (PLOTS)
# ==============================================================================

# --- 6.1. Gráficos da Questão 1 (Double k-means) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva de Convergência de W
axes[0].plot(range(1, len(W_history_star) + 1), W_history_star, marker='o', color='b', linewidth=2)
axes[0].set_title(f"Convergência da Função W (K*={K_star}, H={best_pair[1]})")
axes[0].set_xlabel("Iterações")
axes[0].set_ylabel("Valor de W (Heterogeneidade)")
axes[0].grid(True)

# Heatmap da Matriz de Protótipos G*
sns.heatmap(G_star, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1], cbar=True)
axes[1].set_title(f"Matriz de Protótipos G* (K*={K_star} x H={best_pair[1]})")
axes[1].set_xlabel("Blocos de Variáveis (H)")
axes[1].set_ylabel("Clusters de Objetos (K)")

plt.tight_layout()
plt.show()


# --- 6.2. Gráficos da Questão 2 (Comparação de Modelos e Curvas de Aprendizado) ---
def plot_classification_results(results_orig, results_clust, curves_orig, train_fractions):
    clf_names = list(results_orig.keys())
    
    # 1. Comparativo F1-Score (Com Intervalo de Confiança)
    f1_orig = [np.mean(results_orig[clf]['F1_Measure']) for clf in clf_names]
    f1_clust = [np.mean(results_clust[clf]['F1_Measure']) for clf in clf_names]
    
    err_orig = [stats.sem(results_orig[clf]['F1_Measure']) * 1.96 for clf in clf_names]
    err_clust = [stats.sem(results_clust[clf]['F1_Measure']) * 1.96 for clf in clf_names]

    x = np.arange(len(clf_names))
    width = 0.35

    plt.figure(figsize=(12, 6))
    plt.bar(x - width/2, f1_orig, width, yerr=err_orig, capsize=5, label='Dataset Original (2 Classes)', color='skyblue')
    plt.bar(x + width/2, f1_clust, width, yerr=err_clust, capsize=5, label=f'Dataset Double k-means (K* Classes)', color='salmon')
    
    plt.ylabel('F1-Score Médio (Macro)')
    plt.title('Comparação de Desempenho dos Classificadores (30x10-Folds)')
    plt.xticks(x, clf_names, rotation=15)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 2. Curvas de Aprendizagem (Dataset Original)
    plt.figure(figsize=(12, 6))
    for clf in clf_names:
        mean_test_curve = np.mean(curves_orig[clf]['test'], axis=0)
        plt.plot(train_fractions * 100, mean_test_curve, label=f"{clf} (Teste)")

    plt.title('Curvas de Aprendizagem - Dataset Original (F1-Score no Teste)')
    plt.xlabel('Porcentagem da Base de Treino (%)')
    plt.ylabel('F1-Score Médio')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Executar as visualizações de classificação
plot_classification_results(results_orig, results_clust, curves_orig, train_fractions)

Iniciando varredura dos pares (K, H) com 100 repetições cada...
Par (K=2, H=1) -> Menor W = 256513.75 | Silhueta = 0.2961
Par (K=2, H=2) -> Menor W = 244486.70 | Silhueta = 0.6596
Par (K=3, H=1) -> Menor W = 254414.01 | Silhueta = 0.2515
Par (K=3, H=2) -> Menor W = 239373.66 | Silhueta = 0.2454
Par (K=3, H=3) -> Menor W = 234585.87 | Silhueta = 0.1409
Par (K=4, H=1) -> Menor W = 253278.68 | Silhueta = 0.0484
Par (K=4, H=2) -> Menor W = 233358.70 | Silhueta = 0.2428
Par (K=4, H=3) -> Menor W = 228516.44 | Silhueta = 0.1620
Par (K=4, H=4) -> Menor W = 227058.69 | Silhueta = 0.1536

Melhor combinação (K, H): (2, 2)
Número ótimo de clusters de objetos K* = 2 (Silhueta máxima = 0.6596)

Índice de Rand Corrigido (ARI) para K* = 2: -0.0049

INICIANDO EXPERIMENTO 30x10-FOLDS PARA: Spambase (Original - 2 Classes)
  -> Repetição 1/30...
  -> Repetição 2/30...
  -> Repetição 3/30...
  -> Repetição 4/30...
  -> Repetição 5/30...
  -> Repetição 6/30...
  -> Repetição 7/30...
  -> Repetição 8/30...


/tmp/ipykernel_144808/402548521.py:195: RuntimeWarning: Degrees of freedom <= 0 for slice
  cov = np.cov(X_c, rowvar=False) + 1e-4 * np.eye(self.d_)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


  -> Repetição 2/30...


/tmp/ipykernel_144808/402548521.py:195: RuntimeWarning: Degrees of freedom <= 0 for slice
  cov = np.cov(X_c, rowvar=False) + 1e-4 * np.eye(self.d_)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


  -> Repetição 3/30...


...

  -> Repetição 14/30...


/tmp/ipykernel_144808/402548521.py:195: RuntimeWarning: Degrees of freedom <= 0 for slice
  cov = np.cov(X_c, rowvar=False) + 1e-4 * np.eye(self.d_)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


  -> Repetição 15/30...


/tmp/ipykernel_144808/402548521.py:195: RuntimeWarning: Degrees of freedom <= 0 for slice
  cov = np.cov(X_c, rowvar=False) + 1e-4 * np.eye(self.d_)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/rozyelayny/Documentos/projetoAM/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2901: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


  -> Repetição 16/30...
